# 🧰 quant-kit — Kaggle Benchmark Suite

**Free benchmarks using Kaggle's T4 GPU.**

What this runs automatically:
- ⚡ Speed (PP/TG at 3 context sizes) — for `QUANT_TYPE`
- 📉 Perplexity (WikiText-2) — for **ALL quants** in the repo
- 🧠 lm-eval tasks (TruthfulQA, GPQA, ARC, HellaSwag, GSM8K, Winogrande) — for `QUANT_TYPE`
- 📄 Auto-generates & uploads updated README to HuggingFace — **no laptop commands needed!**

### Setup
1. Add your HuggingFace token as a Kaggle Secret named `HF_TOKEN`
2. Set `HF_REPO` below
3. Enable **GPU T4 x2** in Settings → Accelerator
4. Click **Run All** (~4-6 hours)

In [ ]:
# ── CONFIG — Only edit these ───────────────────────────────────────────
HF_REPO           = "Dhptl/gemma-4-12b-it-GGUF"
ORIGINAL_MODEL_ID = "google/gemma-4-12b-it"
QUANT_TYPE        = "Q4_K_M"

RUN_SPEED   = True
RUN_PPL_ALL = True
RUN_EVAL    = True
EVAL_TASKS  = "truthfulqa_mc2,gpqa_diamond,arc_challenge,hellaswag,gsm8k,winogrande"
# ──────────────────────────────────────────────────────────────────────

In [ ]:
# ── Install dependencies & download llama.cpp ──────────────────────────
import subprocess, os, sys, json
from pathlib import Path

print("Installing Python packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "lm-eval[api]", "psutil",
    "datasets", "jinja2", "requests"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "llama-cpp-python",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"],
    check=True)

# ── Auto-detect latest llama.cpp from GitHub API ─────────────────────
import requests as req
print("Finding latest llama.cpp release...")
resp = req.get(
    "https://api.github.com/repos/ggerganov/llama.cpp/releases/latest",
    headers={"Accept": "application/vnd.github+json"}, timeout=30
)
release  = resp.json()
tag      = release["tag_name"]
assets   = release.get("assets", [])
print(f"Latest release: {tag}")
print(f"Available assets: {[a['name'] for a in assets]}")

# Priority: plain ubuntu-x64 (no vulkan/rocm/openvino/arm variants)
# Accepts both .tar.gz and .zip
def pick_asset(assets):
    EXCLUDE = ["vulkan", "rocm", "openvino", "arm64", "s390x", "adreno", "win", "macos", "android", "xcframework"]
    for ext in [".tar.gz", ".zip"]:
        for a in assets:
            name = a["name"]
            if not ("ubuntu" in name and "x64" in name and name.endswith(ext)):
                continue
            if any(ex in name for ex in EXCLUDE):
                continue
            return a["browser_download_url"], ext
    return None, None

asset_url, ext = pick_asset(assets)
if not asset_url:
    raise RuntimeError(f"No suitable ubuntu-x64 binary found in {tag}. Assets: {[a['name'] for a in assets]}")
print(f"Downloading: {asset_url}")

archive_path = f"/tmp/llama{ext}"
r = req.get(asset_url, stream=True, timeout=300)
r.raise_for_status()
with open(archive_path, "wb") as f:
    for chunk in r.iter_content(chunk_size=65536):
        f.write(chunk)
print(f"Downloaded ({Path(archive_path).stat().st_size / 1e6:.1f} MB)")

# Extract correctly based on format
if ext == ".tar.gz":
    subprocess.run(["tar", "-xzf", archive_path, "-C", "/tmp/llama"], check=True)
else:
    subprocess.run(["unzip", "-q", "-o", archive_path, "-d", "/tmp/llama"], check=True)

Path("/tmp/llama").mkdir(exist_ok=True)
subprocess.run(["chmod", "+x"] + [str(p) for p in Path("/tmp/llama").rglob("llama-*")], check=False)

bench_bin = next(Path("/tmp/llama").rglob("llama-bench"), None)
ppl_bin   = next(Path("/tmp/llama").rglob("llama-perplexity"), None)

if not bench_bin:
    print("All extracted files:", [str(p) for p in Path("/tmp/llama").rglob("*")])
    raise RuntimeError("llama-bench not found after extraction!")

LLAMA_BENCH      = str(bench_bin)
LLAMA_PERPLEXITY = str(ppl_bin) if ppl_bin else None
print(f"llama-bench:      {LLAMA_BENCH}")
print(f"llama-perplexity: {LLAMA_PERPLEXITY}")

In [ ]:
# ── Setup HF token & list all quants ──────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
import os

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)

model_name = HF_REPO.split("/")[1]
base_name  = model_name.replace("-GGUF", "")

all_files  = list(list_repo_files(HF_REPO, token=HF_TOKEN))
all_quants = sorted([f for f in all_files if f.endswith(".gguf") and "F16" not in f])

print(f"Found {len(all_quants)} quants in {HF_REPO}:")
for q in all_quants:
    print(f"  • {q}")

In [ ]:
# ── Download main QUANT_TYPE for speed + lm-eval ──────────────────────
main_gguf_file  = f"{base_name}-{QUANT_TYPE}.gguf"
main_model_path = f"/kaggle/working/{main_gguf_file}"

print(f"Downloading {main_gguf_file}...")
hf_hub_download(repo_id=HF_REPO, filename=main_gguf_file,
                local_dir="/kaggle/working", token=HF_TOKEN)
size_gb = Path(main_model_path).stat().st_size / 1e9
print(f"Ready: {main_gguf_file} ({size_gb:.2f} GB)")

In [ ]:
# ── 1. Speed Benchmark (llama-bench, 3 context sizes) ─────────────────
import json, subprocess

speed_results = []

if RUN_SPEED:
    print("\n" + "="*55)
    print(f"  Speed — {QUANT_TYPE} on T4 GPU")
    print("="*55)
    for ctx in [128, 512, 2048]:
        print(f"  Context {ctx} tokens...")
        cmd = [LLAMA_BENCH, "-m", main_model_path,
               "-ngl", "99", "-p", str(ctx), "-n", "128", "-r", "3", "--output", "json"]
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
        pp_tps, tg_tps = None, None
        if r.returncode == 0 and r.stdout.strip():
            try:
                for entry in json.loads(r.stdout):
                    if isinstance(entry, dict):
                        if entry.get("n_prompt", 0) > 0 and entry.get("n_gen", 0) == 0:
                            pp_tps = round(float(entry.get("avg_ts", 0)), 2)
                        elif entry.get("n_gen", 0) > 0 and entry.get("n_prompt", 0) == 0:
                            tg_tps = round(float(entry.get("avg_ts", 0)), 2)
            except Exception: pass
        speed_results.append({"context": ctx, "pp_tok_s": pp_tps, "tg_tok_s": tg_tps})
        print(f"    TG={tg_tps} tok/s  PP={pp_tps} tok/s")
    print("Speed done!")

In [ ]:
# ── 2. Perplexity for ALL quants ──────────────────────────────────────
import re
import datasets as ds_lib

ppl_all = {}

if RUN_PPL_ALL:
    print("\n" + "="*55)
    print("  Perplexity — ALL quants on WikiText-2")
    print("="*55)

    if LLAMA_PERPLEXITY is None:
        print("WARNING: llama-perplexity not in this release. Skipping PPL.")
    else:
        wiki_path = "/kaggle/working/wiki.test.raw"
        if not Path(wiki_path).exists():
            dataset = ds_lib.load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
            with open(wiki_path, "w") as f:
                f.write("\n".join(dataset["text"]))
            print("WikiText-2 ready")

        for gguf_filename in all_quants:
            quant_type = gguf_filename.replace(".gguf", "").split("-")[-1]
            local_path = f"/kaggle/working/{gguf_filename}"

            if not Path(local_path).exists():
                print(f"  Downloading {gguf_filename}...")
                hf_hub_download(repo_id=HF_REPO, filename=gguf_filename,
                                local_dir="/kaggle/working", token=HF_TOKEN)

            print(f"  PPL: {quant_type}...")
            cmd = [LLAMA_PERPLEXITY, "-m", local_path,
                   "-f", wiki_path, "-c", "512", "-ngl", "99"]
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)

            ppl = None
            for line in reversed(r.stderr.splitlines()):
                if "Final estimate" in line and "PPL" in line:
                    m = re.search(r"PPL\s*=\s*([\d.]+)", line)
                    if m:
                        ppl = float(m.group(1))
                        break
            ppl_all[quant_type] = ppl
            print(f"    {quant_type}: PPL = {ppl}")

            if local_path != main_model_path:
                Path(local_path).unlink(missing_ok=True)

    print("Perplexity done!")

In [ ]:
# ── 3. lm-eval downstream benchmarks ─────────────────────────────────
eval_results = {}

if RUN_EVAL:
    print("\n" + "="*55)
    print(f"  lm-eval — {QUANT_TYPE}: {EVAL_TASKS}")
    print("="*55)

    results_dir = Path("/kaggle/working/eval_results")
    results_dir.mkdir(exist_ok=True)

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model", "gguf",
        "--model_args", f"pretrained={main_model_path}",
        "--tasks", EVAL_TASKS,
        "--output_path", str(results_dir),
        "--batch_size", "auto",
        "--device", "cuda",
    ]
    print("Running (2-4 hours)...")
    subprocess.run(cmd, text=True, timeout=18000)

    for result_file in results_dir.glob("**/*.json"):
        if "results" in result_file.name:
            with open(result_file) as f:
                data = json.load(f)
            for task, metrics in data.get("results", {}).items():
                score = (
                    metrics.get("acc_norm,none") or
                    metrics.get("acc,none") or
                    metrics.get("exact_match,none")
                )
                if score is not None:
                    eval_results[task] = round(score * 100, 2)
                    print(f"  {task}: {eval_results[task]}%")
            break
    print("lm-eval done!")

In [ ]:
# ── 4. Save + upload results JSON ─────────────────────────────────────
from huggingface_hub import HfApi

output = {
    "model":          HF_REPO,
    "quant":          QUANT_TYPE,
    "platform":       "Kaggle T4 GPU",
    "speed":          speed_results,
    "perplexity_all": ppl_all,
    "perplexity":     ppl_all.get(QUANT_TYPE),
    "benchmarks":     eval_results,
}

result_file = f"/kaggle/working/kaggle_results_{QUANT_TYPE}.json"
with open(result_file, "w") as f:
    json.dump(output, f, indent=2)

api.upload_file(
    path_or_fileobj=result_file,
    path_in_repo=f"kaggle_results_{QUANT_TYPE}.json",
    repo_id=HF_REPO, repo_type="model",
    commit_message=f"Add Kaggle benchmark results ({QUANT_TYPE})"
)
print(f"Results JSON uploaded to {HF_REPO}")

In [ ]:
# ── 5. Auto-generate & upload README — no laptop commands needed! ──────
from datetime import datetime
from huggingface_hub import ModelCard

TASK_META = {
    "truthfulqa_mc2": ("TruthfulQA",   "Resistance to hallucination"),
    "gpqa_diamond":   ("GPQA Diamond",  "Hard science reasoning (PhD-level)"),
    "arc_challenge":  ("ARC Challenge", "Grade-school science reasoning"),
    "hellaswag":      ("HellaSwag",     "Common sense completion"),
    "gsm8k":          ("GSM8K",         "Grade-school math word problems"),
    "winogrande":     ("Winogrande",    "Commonsense pronoun resolution"),
}

try:
    license_ = ModelCard.load(ORIGINAL_MODEL_ID).data.get("license", "other")
except Exception:
    license_ = "other"

ppl_table = "| Quant | Perplexity ↓ | Notes |\n|---|---|---|\n"
best_ppl  = min((v for v in ppl_all.values() if v), default=None)
for qt, ppl in sorted(ppl_all.items()):
    note = "✅ Best" if (ppl and best_ppl and ppl == best_ppl) else ""
    ppl_table += f"| `{qt}` | `{ppl}` | {note} |\n"

bench_table = "| Benchmark | Score | What it measures |\n|---|---|---|\n"
for task, score in eval_results.items():
    name, desc = TASK_META.get(task, (task, ""))
    bench_table += f"| **{name}** | `{score}%` | {desc} |\n"

speed_table = "| Context | Token Generation | Prompt Processing |\n|---|---|---|\n"
for r in speed_results:
    tg = f"{r['tg_tok_s']} tok/s" if r.get('tg_tok_s') else "—"
    pp = f"{r['pp_tok_s']} tok/s" if r.get('pp_tok_s') else "—"
    speed_table += f"| {r['context']} tokens | {tg} | {pp} |\n"

readme = f"""---
license: {license_}
base_model: {ORIGINAL_MODEL_ID}
pipeline_tag: text-generation
tags:\n  - gguf\n  - quantized\n  - text-generation
language:\n  - en
---

<div align=\"center\">

# {model_name}

Quantized GGUF versions of [{ORIGINAL_MODEL_ID}](https://huggingface.co/{ORIGINAL_MODEL_ID}).  
Works with llama.cpp, Ollama, LM Studio, and any GGUF-compatible runtime.

*Benchmarked on Kaggle T4 GPU · {datetime.now().strftime("%B %d, %Y")} · Built with [quant-kit](https://github.com/DhruvalPtl/quant-kit)*

</div>

---

## ⚖️ Quality Degradation (Perplexity on WikiText-2)

Lower = closer to original FP16 quality.

{ppl_table}

---

## 🧠 Downstream Benchmarks — `{QUANT_TYPE}` on T4 GPU

*Evaluated using [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)*

{bench_table}

---

## ⚡ Speed — `{QUANT_TYPE}` on Kaggle T4 GPU

{speed_table}

---

## 🚀 How to Use

### llama.cpp
```bash
./llama-cli -m {base_name}-Q4_K_M.gguf -p \"Your prompt\" -n 512
```

### Python
```python
from llama_cpp import Llama
llm = Llama(model_path=\"./{base_name}-Q4_K_M.gguf\", n_gpu_layers=-1)
print(llm(\"Tell me about AI\", max_tokens=256)[\"choices\"][0][\"text\"])
```

---
*Quantized with [quant-kit](https://github.com/DhruvalPtl/quant-kit)*
"""

readme_path = "/kaggle/working/README.md"
with open(readme_path, "w") as f:
    f.write(readme)

api.upload_file(
    path_or_fileobj=readme_path,
    path_in_repo="README.md",
    repo_id=HF_REPO, repo_type="model",
    commit_message="Auto-update README with Kaggle benchmark results"
)

print(f"\n✅ README uploaded → https://huggingface.co/{HF_REPO}")
print("🎉 All done! No laptop commands needed.")
print(json.dumps(output, indent=2))